# RSNA Knee: a visual guide to the data

This notebook is a tour of the data, with pictures. It covers what a study looks like, how the
series differ, what the reports say, and how few studies carry expert labels. Everything runs on
CPU in a few minutes, using only the competition files.

**Contents**
1. The task in one table
2. Studies and series: planes, fat suppression, fluid weighting
3. DICOM headers: scanners, field strength, slice counts, resolution
4. Looking at a knee: every series of one study
5. Scrolling through a series (animated)
6. The reports: languages, length, what they mention
7. The 58 expert-labelled studies: prevalence and co-occurrence
8. Findings side by side: positive vs negative gold studies
9. Practical notes for building a model

In [ ]:
import os, glob, re, io, base64, random, warnings
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import HTML, display
import pydicom
warnings.filterwarnings("ignore")
random.seed(0); np.random.seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.labelsize": 10})

def find_root():
    for d in ("/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"):
        if os.path.isfile(os.path.join(d, "train.csv")):
            return d
    return next(d for d, _, f in os.walk("/kaggle/input") if "train_series.csv" in f)

ROOT = find_root()
IMG = next(p for p in (f"{ROOT}/train_series", f"{ROOT}/train_images") if os.path.isdir(p))
print(ROOT); print(sorted(os.listdir(ROOT)))
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
train = pd.read_csv(f"{ROOT}/train.csv", dtype={"StudyInstanceUID": str})
tser = pd.read_csv(f"{ROOT}/train_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
test = pd.read_csv(f"{ROOT}/test.csv", dtype={"StudyInstanceUID": str})
sser = pd.read_csv(f"{ROOT}/test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
gold = train.dropna(subset=LAB).copy()
print(f"train studies {len(train):,} | train series {len(tser):,} | gold-labelled {len(gold)} | "
      f"test studies visible here {len(test):,} (the hidden test set is ~1,300 studies)")

## 1. The task in one table

| | |
|---|---|
| Unit of prediction | one **study** (a knee MRI exam), 12 binary findings |
| Findings | ACL, MCL, medial / lateral meniscus, medial / lateral / patellofemoral OA, effusion, synovitis, Baker's cyst, contusion, fracture |
| Metric | mean ROC-AUC over the 12 findings, so only the **ranking** within each column matters |
| Train labels | a free-text radiology **report** for every study; expert 0/1 labels for only a small subset |
| Test input | images only, no report |

So the real problem has two halves: turn reports into training targets, then teach an image model
to reproduce them. The test set has no report, so everything must come from pixels.

In [ ]:
print(train.columns.tolist())
display(train.head(3)[["StudyInstanceUID", "Report"] + LAB[:4]])
display(tser.head())

## 2. Studies and series

Each study is a folder of **series**. Each series is one MRI sequence: a stack of 2D slices in one
plane, with one contrast. `train_series.csv` describes every series with three columns:

* `Anatomical_Plane`: sagittal, coronal or axial
* `Fat_Suppression`: 1 if fat is suppressed (fat dark, fluid and oedema stand out)
* `Fluid_Sensitive`: 1 for fluid-bright weightings (PD / T2-like); 0 is typically T1-like

The plane × fat × fluid combination tells you what a series is good for. Fat-suppressed fluid
sequences show bone bruises and effusion. Sagittal series show the cruciate ligaments and the
menisci. Coronal series show the MCL and the joint compartments.

In [ ]:
for name, df in (("train", tser), ("test", sser)):
    per = df.groupby("StudyInstanceUID").size()
    print(f"{name}: series per study mean {per.mean():.2f}, median {per.median():.0f}, "
          f"min {per.min()}, max {per.max()}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
per = tser.groupby("StudyInstanceUID").size()
ax[0].hist(per, bins=np.arange(per.min() - .5, per.max() + 1.5), color="#4C72B0")
ax[0].set(title="Series per study (train)", xlabel="series", ylabel="studies")

combo = (tser["Anatomical_Plane"] + " | fat " + tser["Fat_Suppression"].astype(str)
         + " | fluid " + tser["Fluid_Sensitive"].astype(str))
vc = combo.value_counts()
ax[1].barh(vc.index[::-1], vc.values[::-1], color="#55A868")
ax[1].set(title="Series types (train)", xlabel="series")

have = tser.groupby(["StudyInstanceUID", "Anatomical_Plane"]).size().unstack(fill_value=0)
planes = ["Sagittal", "Coronal", "Axial"]
for i, (pl, col) in enumerate(zip(planes, ["#C44E52", "#8172B2", "#DD8452"])):
    vc = have[pl].clip(upper=4).value_counts().reindex([0, 1, 2, 3, 4], fill_value=0)
    ax[2].bar(np.arange(5) + (i - 1) * .27, vc.values, width=.27, color=col, label=pl)
ax[2].set_xticks(range(5)); ax[2].set_xticklabels(["0", "1", "2", "3", "4+"])
ax[2].set(title="Series per plane, per study", xlabel="series in that plane", ylabel="studies"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Which plane x fat-suppression "slots" does a typical study fill? (share of studies)
slot = tser.copy()
slot["slot"] = tser["Anatomical_Plane"] + np.where(tser["Fat_Suppression"] == 1, " fat-sat", " no fat-sat")
pres = slot.groupby(["StudyInstanceUID", "slot"]).size().unstack(fill_value=0) > 0
order = ["Sagittal fat-sat", "Sagittal no fat-sat", "Coronal fat-sat", "Coronal no fat-sat",
         "Axial fat-sat", "Axial no fat-sat"]
pres = pres.reindex(columns=order, fill_value=False)
pat = pres.apply(lambda r: " + ".join(c.replace(" fat-sat", "+FS").replace(" no", "")
                                       .replace("Sagittal", "SAG").replace("Coronal", "COR")
                                       .replace("Axial", "AX") for c in order if r[c]), axis=1)
print("share of studies with each slot:"); display(pres.mean().round(3).to_frame("share"))
print("most common slot patterns:"); display(pat.value_counts().head(8).to_frame("studies"))

## 3. DICOM headers

Headers are cheap to read with `stop_before_pixels=True`. Below is a random sample of series, one
header per series plus the slice count.

In [ ]:
TAGS = ["Manufacturer", "MagneticFieldStrength", "SliceThickness", "Rows", "Columns",
        "RepetitionTime", "EchoTime", "SeriesDescription"]
def series_dir(study, series, root=IMG):
    return f"{root}/{study}/{series}"

samp = tser.sample(600, random_state=0)
rows = []
for r in samp.itertuples():
    files = glob.glob(series_dir(r.StudyInstanceUID, r.SeriesInstanceUID) + "/*.dcm")
    if not files:
        continue
    try:
        h = pydicom.dcmread(files[0], stop_before_pixels=True)
    except Exception:
        continue
    d = {t: getattr(h, t, None) for t in TAGS}
    ps = getattr(h, "PixelSpacing", None)
    d.update(n_slices=len(files), pixel_mm=float(ps[0]) if ps else np.nan,
             plane=r.Anatomical_Plane, fat=r.Fat_Suppression, fluid=r.Fluid_Sensitive,
             tsyntax=str(getattr(h.file_meta, "TransferSyntaxUID", "?").name))
    rows.append(d)
hdr = pd.DataFrame(rows)
for c in ["MagneticFieldStrength", "SliceThickness", "RepetitionTime", "EchoTime"]:
    hdr[c] = pd.to_numeric(hdr[c], errors="coerce")
hdr["Manufacturer"] = hdr["Manufacturer"].fillna("?").str.upper().str.extract(r"(SIEMENS|GE|PHILIPS|TOSHIBA|CANON|HITACHI|FUJI)", expand=False).fillna("OTHER")
print(f"read {len(hdr)} series headers")

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
hdr["Manufacturer"].value_counts().plot.bar(ax=ax[0, 0], color="#4C72B0", rot=0, title="Manufacturer")
hdr["MagneticFieldStrength"].round(1).value_counts().sort_index().plot.bar(ax=ax[0, 1], color="#8172B2", rot=0, title="Field strength (T)")
ax[0, 2].hist(hdr["n_slices"].clip(upper=80), bins=40, color="#55A868"); ax[0, 2].set_title("Slices per series (clipped at 80)")
ax[1, 0].hist(hdr["pixel_mm"].clip(upper=1.2), bins=40, color="#C44E52"); ax[1, 0].set_title("In-plane pixel size (mm)")
ax[1, 1].hist(hdr["SliceThickness"].clip(upper=8), bins=30, color="#DD8452"); ax[1, 1].set_title("Slice thickness (mm)")
for (fl, g), col in zip(hdr.groupby("fluid"), ["#4C72B0", "#C44E52"]):
    ax[1, 2].scatter(g["EchoTime"], g["RepetitionTime"], s=8, alpha=.5, c=col, label=f"Fluid_Sensitive={fl}")
ax[1, 2].set(xscale="log", yscale="log", xlabel="TE (ms)", ylabel="TR (ms)", title="TR vs TE by fluid flag"); ax[1, 2].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(hdr["tsyntax"].value_counts().to_frame("series (sample)"))

Things to notice:

* Studies come from several vendors and both 1.5 T and 3 T scanners. Contrast and noise vary a
  lot, so normalise each series on its own (percentiles work well) rather than using fixed
  windows.
* In-plane pixel size runs from about 0.15 mm to 0.6 mm (a factor of 3 to 4). Crop by **millimetres**, not pixels, if you want the
  knee to be the same size in every input.
* `Fluid_Sensitive` mostly separates in TR/TE space (long TR and longer TE = fluid-bright), with some overlap.
* Most series are around 30 slices of about 3 mm, but a few are thin-slice 3D acquisitions with 100+ slices.
* Several transfer syntaxes (compressed JPEG Lossless / JPEG 2000) appear, so make sure your
  decoder handles all of them.

## 4. Looking at a knee

Slices inside a series are **not** guaranteed to be in file-name order. Sort them by position
along the slice normal (`ImagePositionPatient` · normal of `ImageOrientationPatient`).

In [ ]:
def ordered_files(sdir):
    recs = []
    for f in glob.glob(sdir + "/*.dcm"):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = np.array(h.ImageOrientationPatient, float)
            pos = float(np.dot(np.array(h.ImagePositionPatient, float), np.cross(iop[:3], iop[3:])))
        except Exception:
            pos = float(getattr(h, "InstanceNumber", 0) or 0)
        recs.append((pos, f))
    return [f for _, f in sorted(recs)]

def read_px(f):
    d = pydicom.dcmread(f)
    a = d.pixel_array.astype(np.float32)
    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a
    return a

def norm(a, lo=2, hi=98):
    p0, p1 = np.percentile(a, [lo, hi])
    return np.clip((a - p0) / max(p1 - p0, 1e-6), 0, 1)

def load_series(study, series):
    fs = ordered_files(series_dir(study, series))
    out = []
    for f in fs:
        try:
            out.append(read_px(f))
        except Exception:
            return None   # undecodable (missing codec) -> skip this series
    return out

def series_label(r):
    return (f"{r['Anatomical_Plane']}  {'fat-sat' if r['Fat_Suppression'] == 1 else 'no fat-sat'}  "
            f"{'fluid' if r['Fluid_Sensitive'] == 1 else 'T1-like'}")

def show_study(study, title=None, n_cols=3):
    rows = tser[tser.StudyInstanceUID == study].sort_values(["Anatomical_Plane", "Fat_Suppression"]).to_dict("records")
    vols = [(r, load_series(study, r["SeriesInstanceUID"])) for r in rows]
    vols = [(r, v) for r, v in vols if v]
    if not vols:
        return False
    fig, axs = plt.subplots(len(vols), n_cols, figsize=(3.2 * n_cols, 3.2 * len(vols)))
    axs = np.atleast_2d(axs)
    for (r, v), axr in zip(vols, axs):
        for j, q in enumerate(np.linspace(.3, .7, n_cols)):
            ax = axr[j]; k = int(q * (len(v) - 1))
            ax.imshow(norm(v[k]), cmap="gray"); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"slice {k + 1}/{len(v)}", fontsize=8)
        axr[0].set_ylabel(series_label(r), fontsize=9)
    fig.suptitle(title or f"study {study[-12:]}", y=1.0); plt.tight_layout(); plt.show()
    return True

for s in gold.sample(frac=1, random_state=1).StudyInstanceUID:
    pos = [c for c in LAB if gold.set_index("StudyInstanceUID").loc[s, c] == 1]
    if show_study(s, f"Gold study ...{s[-8:]} | positive: {', '.join(pos) or 'none'}"):
        break

Each row is a series, at 30 %, 50 % and 70 % through the stack. Some studies hold two
series of the same type (here, two coronal fat-sat series), and some are thin-slice 3D
acquisitions with very different slice counts. The same knee looks very
different from one sequence to the next. Fat-suppressed fluid series (bright fluid, dark fat) are
where effusion, bone bruise (contusion) and Baker's cysts stand out.

## 5. Scrolling through a series

In [ ]:
from PIL import Image

def gif_html(vol, size=256, fps=12, title=""):
    frames = [Image.fromarray((norm(a) * 255).astype(np.uint8)).resize((size, size)) for a in vol]
    buf = io.BytesIO()
    frames[0].save(buf, format="GIF", save_all=True, append_images=frames[1:], duration=int(1000 / fps), loop=0)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return (f'<div style="display:inline-block;margin:4px;text-align:center"><img src="data:image/gif;base64,{b64}"/>'
            f'<div style="font-size:12px">{title}</div></div>')

def first_ok(study, plane, fat=None):
    rows = tser[(tser.StudyInstanceUID == study) & (tser.Anatomical_Plane == plane)]
    if fat is not None and (rows.Fat_Suppression == fat).any():
        rows = rows[rows.Fat_Suppression == fat]
    for r in rows.to_dict("records"):
        v = load_series(study, r["SeriesInstanceUID"])
        if v:
            return r, v
    return None, None

html = []
for s in gold.sample(frac=1, random_state=2).StudyInstanceUID:
    trip = [first_ok(s, p, 1) for p in ("Sagittal", "Coronal", "Axial")]
    if all(v for _, v in trip):
        html = [gif_html(v, title=series_label(r)) for r, v in trip]
        break
display(HTML("".join(html)))

## 6. The reports

Every train study has a report. They are written in **many languages**, and the test set has no
reports at all. The rough language guess below uses a few common words per language. It isn't a
real language detector, but it is enough to see the mix.

In [ ]:
CUES = {
    "Spanish": r"\b(rodilla|menisco|derrame|rotura|ligamento cruzado)\b",
    "Portuguese": r"\b(joelho|menisco|derrame articular|ligamento cruzado|rotura)\b.*\b(n[aã]o|com|sem)\b",
    "German": r"\b(knie|meniskus|gelenkerguss|befund|kreuzband)\b",
    "Dutch": r"\b(knie|meniscus|bevindingen|kruisband|vocht)\b",
    "French": r"\b(genou|m[ée]nisque|[ée]panchement|ligament crois[ée])\b",
    "Italian": r"\b(ginocchio|menisco|versamento|legamento crociato)\b",
    "English": r"\b(knee|meniscus|effusion|tear|cruciate)\b",
    "Turkish": r"\b(diz|menisk|efüzyon|bağ)\b",
}
def guess_lang(t):
    t = str(t).lower()
    if re.search(r"[぀-ヿ一-鿿]", t): return "CJK"
    if re.search(r"[Ѐ-ӿ]", t): return "Cyrillic"
    if re.search(r"[؀-ۿ]", t): return "Arabic"
    if re.search(r"[Ͱ-Ͽ]", t): return "Greek"
    hits = {k: len(re.findall(v, t)) for k, v in CUES.items()}
    k = max(hits, key=hits.get)
    return k if hits[k] else "other"

train["lang"] = train["Report"].map(guess_lang)
train["rep_len"] = train["Report"].fillna("").str.len()
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
train["lang"].value_counts().plot.bar(ax=ax[0], color="#4C72B0", rot=30, title="Report language (keyword guess)")
ax[1].hist(train["rep_len"].clip(upper=4000), bins=50, color="#55A868")
ax[1].set(title="Report length (characters)", xlabel="chars")
plt.tight_layout(); plt.show()
for lg in train["lang"].value_counts().index[:4]:
    t = train.loc[train.lang == lg, "Report"].iloc[0]
    print(f"--- {lg}: {str(t)[:260]}...")

Things to notice:

* Reports differ in length and structure. Some have a findings section and an impression; some
  are a few lines.
* A report mentions what the radiologist cared about. A finding that is never mentioned is
  **not** the same as a confirmed negative. This is the central label-noise problem here: a
  silent report is weak evidence of absence.
* Negation is everywhere ("no tear", "sin derrame", "kein Erguss"), so plain keyword matching
  over-counts positives.

In [ ]:
# How often do simple multilingual keywords appear? (crude, negation-blind; for scale only)
KW = {
    "Effusion": r"effusion|derrame|erguss|epanchement|épanchement|versamento|vocht|hydrops",
    "Baker's": r"baker",
    "ACL": r"\bacl\b|\blca\b|anterior cruciate|cruzado anterior|vorderes kreuzband|voorste kruisband|crois[ée] ant",
    "Fracture": r"fractur|fraktur|breuk",
    "Contusion": r"contusi|bone bruise|kontusion|bone marrow edema|edema [óo]seo|knochenmark[öo]dem|beenmergoedeem",
}
rep = train["Report"].fillna("").str.lower()
display(pd.DataFrame({k: [rep.str.contains(v).mean()] for k, v in KW.items()}, index=["share of reports mentioning"]).round(3))

## 7. The expert-labelled studies

Only a small subset of the training studies comes with expert 0/1 labels. They are worth looking
at closely: they show how common each finding is, and they're the only clean yardstick for any
report-to-label method you build.

In [ ]:
G = gold.set_index("StudyInstanceUID")[LAB].astype(int)
fig = plt.figure(figsize=(15, 4.6)); gs = gridspec.GridSpec(1, 3, width_ratios=[1.1, 1.3, .8])
ax0 = fig.add_subplot(gs[0]); prev = G.mean().sort_values()
ax0.barh(prev.index, prev.values, color="#4C72B0")
for i, v in enumerate(prev.values): ax0.text(v + .01, i, f"{v:.0%} ({int(v * len(G))})", va="center", fontsize=8)
ax0.set(title=f"Prevalence in {len(G)} gold studies", xlim=(0, 1.15))
ax1 = fig.add_subplot(gs[1]); C = G.corr().values; np.fill_diagonal(C, np.nan)
im = ax1.imshow(C, cmap="RdBu_r", vmin=-.6, vmax=.6)
ax1.set_xticks(range(12)); ax1.set_xticklabels(LAB, rotation=60, ha="right", fontsize=8)
ax1.set_yticks(range(12)); ax1.set_yticklabels(LAB, fontsize=8); ax1.set_title("Correlation between findings (gold)")
plt.colorbar(im, ax=ax1, fraction=.046)
ax2 = fig.add_subplot(gs[2]); k = G.sum(1)
ax2.hist(k, bins=np.arange(-.5, 12.5), color="#55A868"); ax2.set(title="Positive findings per study", xlabel="count")
plt.tight_layout(); plt.show()

Things to notice:

* Degenerative findings cluster: medial OA, medial meniscus and patellofemoral OA rise together,
  effusion and synovitis go together, and ACL tears come with bone contusions (the classic
  pivot-shift bruise pattern). A model that shares information between heads can use this.
* MCL and lateral OA have only about 10 positives among the gold studies. With so few cases, an AUC
  measured on them is **very noisy**: a single study can move one finding's AUC by several
  points. Use the gold set for sanity checks, not fine model selection.

## 8. Findings side by side

In [ ]:
def mid_slice(study, plane, fat=1):
    r, v = first_ok(study, plane, fat)
    if not v: return None, None
    return r, norm(v[len(v) // 2])

def pos_neg(finding, plane, fat=1, n=3):
    pos = G.index[G[finding] == 1].tolist(); neg = G.index[G[finding] == 0].tolist()
    random.shuffle(pos); random.shuffle(neg)
    out = {"positive": [], "negative": []}
    for key, pool in (("positive", pos), ("negative", neg)):
        for s in pool:
            r, a = mid_slice(s, plane, fat)
            if a is not None: out[key].append(a)
            if len(out[key]) == n: break
    fig, axs = plt.subplots(2, n, figsize=(3 * n, 6.2))
    for i, key in enumerate(("positive", "negative")):
        for j in range(n):
            ax = axs[i, j]; ax.set_xticks([]); ax.set_yticks([])
            if j < len(out[key]): ax.imshow(out[key][j], cmap="gray")
            if j == 0: ax.set_ylabel(key, fontsize=11)
    fig.suptitle(f"{finding}: {plane} {'fat-sat' if fat else ''} mid slice, gold studies", y=1.0)
    plt.tight_layout(); plt.show()

pos_neg("Effusion", "Sagittal", 1)
pos_neg("Medial OA", "Coronal", 1)
pos_neg("Contusion", "Sagittal", 1)

A single middle slice is often enough for effusion: there is a bright fluid pocket above the
patella. Contusion and meniscal tears are local and can sit in any slice, which is why most strong
solutions look at **many slices** (2.5D stacks or multiple-instance pooling) rather than one image.

## 9. Practical notes

1. **Read the geometry, not the file names.** Order slices by position; crop by millimetres
   using `PixelSpacing`.
2. **Normalise per series** (for example 2nd to 98th percentile). Vendors and field strengths
   differ.
3. **Choose series by type.** `Anatomical_Plane`, `Fat_Suppression` and `Fluid_Sensitive` let you
   pick, for example, one sagittal fat-sat, one coronal fat-sat and one axial series per study.
   Fall back to whatever exists when a slot is missing.
4. **Labels come from reports**, and a silent report is not a negative. How you handle findings
   the report never mentions matters as much as the image model.
5. **Only rank order is scored.** Ensembles should average ranks or calibrated probabilities per
   column. Monotone rescaling doesn't change AUC.
6. **Treat the gold subset as a noisy yardstick.** It is small; use cross-validation on report
   labels as the main signal.

If this helped you get oriented, an upvote helps other newcomers find it. Questions and
corrections are welcome in the comments.